## Manejo de Errores y Reintentos

Este notebook incluye las siguientes mejoras para el manejo de errores:

### 1. Lógica de Reintentos
* Las peticiones a la API se reintentan automáticamente hasta 3 veces
* Implementa backoff exponencial (1s, 2s, 4s) entre intentos
* Timeout de 30 segundos por petición

### 2. Almacenamiento de Errores
* Los errores se guardan en `/Volumes/workspace/formula_1/formula_1/errors/`
* Particionados por año y mes: `year=YYYY/month=MM/`
* Incluyen timestamp, endpoint, parámetros y número de intentos
* Formato JSON consistente con los datos extraídos

### 3. Manejo Robusto
* Si una extracción falla, se registra el error y se continúa con el siguiente elemento
* Permite identificar problemas específicos sin detener todo el proceso

In [0]:
from datetime import datetime, timedelta
import json
import logging
import time
import requests
# from src.scripts.extract_data import ExtractData

In [0]:
base_url = "https://api.openf1.org/v1"
base_volume = "/Volumes/workspace/formula_1/formula_1"
year = None
month = None

In [0]:
def get_api_data(endpoint: str, params: dict = None, delay: int = 0, max_retries: int = 3):
    """Obtiene datos de la API con lógica de reintentos exponenciales"""
    logging.info(f"GET {endpoint} | params={params}")
    
    for attempt in range(max_retries):
        try:
            if delay > 0:
                time.sleep(delay)
            
            response = requests.get(endpoint, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.RequestException as e:
            wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
            
            if attempt < max_retries - 1:
                logging.warning(f"Request failed (attempt {attempt + 1}/{max_retries}): {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                logging.error(f"Request failed after {max_retries} attempts: {e}")
                return {"error": str(e), "endpoint": endpoint, "params": params, "attempts": max_retries}

In [0]:
def get_meetings(params: dict):
    return get_api_data(f"{base_url}/meetings", params)

def get_sessions(meeting_key: str):
    return get_api_data(f"{base_url}/sessions", {"meeting_key": meeting_key})

def get_drivers(meeting_key: str, session_key: str):
    return get_api_data(f"{base_url}/drivers", {
        "meeting_key": meeting_key,
        "session_key": session_key
    })

def get_laps(meeting_key: str, session_key: str, driver_number: str):
    return get_api_data(f"{base_url}/laps", {
        "meeting_key": meeting_key,
        "session_key": session_key,
        "driver_number": driver_number
    })

def get_cars(session_key: str, driver_number: str, speed: int):
    return get_api_data(f"{base_url}/car_data", {
        # "meeting_key": meeting_key,
        "session_key": session_key,
        "driver_number": driver_number,
        "speed>": speed
    })

In [0]:
def save_json(data, entity, year, month, filename):
    # 1. Construir la ruta (Usamos month:02d para que sea 03 en vez de 3)
    partition_path = f"{base_volume}/{entity}/year={year}/month={month:02d}"
    
    # 2. Asegurar que el directorio existe (Nativo de Databricks)
    dbutils.fs.mkdirs(partition_path)
    
    # 3. RUTA DIRECTA (Sin /dbfs)
    # Los Volumes en Unity Catalog ya están mapeados al sistema de archivos local del cluster
    full_path = f"{partition_path}/{filename}"
    
    try:
        with open(full_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"✅ Guardado exitoso: {full_path}")
    except Exception as e:
        print(f"❌ Error al guardar {filename}: {e}")
        
        # 4. ÚLTIMO RECURSO: Si falla lo anterior, intentar con /dbfs explícito
        # (Solo para clusters antiguos o configuraciones específicas)
        try:
            alt_path = f"/dbfs{full_path}"
            with open(alt_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
            print(f"✅ Guardado (usando /dbfs): {alt_path}")
        except Exception as alt_e:
            print(f"❌ Fallo total en {filename}: {alt_e}")

def save_error_json(error_data, year, month, filename):
    """Guarda errores en la carpeta errors del volumen"""
    partition_path = f"{base_volume}/errors/year={year}/month={month:02d}"
    
    # Asegurar que el directorio existe
    dbutils.fs.mkdirs(partition_path)
    
    full_path = f"{partition_path}/{filename}"
    
    # Agregar timestamp al error
    error_data["timestamp"] = datetime.now().isoformat()
    
    try:
        with open(full_path, "w", encoding="utf-8") as f:
            json.dump(error_data, f, ensure_ascii=False, indent=4)
        print(f"🔴 Error guardado: {full_path}")
    except Exception as e:
        print(f"❌ Error al guardar error en {filename}: {e}")
        try:
            alt_path = f"/dbfs{full_path}"
            with open(alt_path, "w", encoding="utf-8") as f:
                json.dump(error_data, f, ensure_ascii=False, indent=4)
            print(f"🔴 Error guardado (usando /dbfs): {alt_path}")
        except Exception as alt_e:
            print(f"❌ Fallo total al guardar error en {filename}: {alt_e}")

In [0]:
def extract_meetings(date_start: datetime, date_end: datetime):
    params = {
        "date_start>": date_start.strftime("%Y-%m-%d"),
        "date_start<": date_end.strftime("%Y-%m-%d")
    }
    year = date_start.year
    month = date_start.month
    period = date_start.strftime("%Y-%m-%d")[0:7]
    meetings = get_meetings(params)
    
    if isinstance(meetings, __builtins__.dict) and "error" in meetings:
        save_error_json(meetings, year, month, f"error_meetings_{period}.json")
        return []
    
    save_json(meetings, "meetings", year, month, f"meetings_{period}.json")
    return meetings

def extract_sessions(meeting_key, period):
    sessions = get_sessions(meeting_key)
    
    if isinstance(sessions, __builtins__.dict) and "error" in sessions:
        save_error_json(sessions, year, month, f"error_sessions_by_meeting_{meeting_key}___{period}.json")
        return []
    
    save_json(sessions, "sessions", year, month, f"sessions_by_meeting_{meeting_key}___{period}.json")
    return sessions

def extract_drivers(meeting_key, session_key, period):
    drivers = get_drivers(meeting_key, session_key)
    
    if isinstance(drivers, __builtins__.dict) and "error" in drivers:
        save_error_json(drivers, year, month, f"error_drivers_by_meeting_{meeting_key}__by_session_{session_key}___{period}.json")
        return []
    
    save_json(drivers, "drivers", year, month, f"drivers_by_meeting_{meeting_key}__by_session_{session_key}___{period}.json")
    return drivers

def extract_laps(meeting_key, session_key, driver_number, period):
    laps = get_laps(meeting_key, session_key, driver_number)
    
    if isinstance(laps, __builtins__.dict) and "error" in laps:
        save_error_json(laps, year, month, f"error_laps_by_meeting_{meeting_key}__by_session_{session_key}__by_driver_{driver_number}___{period}.json")
        return []
    
    save_json(laps, "laps", year, month, f"laps_by_meeting_{meeting_key}__by_session_{session_key}__by_driver_{driver_number}___{period}.json")
    return laps

def extract_cars(session_key, driver_number, period):
    cars = get_cars(session_key, driver_number, 310)
    
    if isinstance(cars, __builtins__.dict) and "error" in cars:
        save_error_json(cars, year, month, f"error_cars_by_session_{session_key}__by_driver_{driver_number}___{period}.json")
        return []
    
    save_json(cars, "cars", year, month, f"cars_by_session_{session_key}__by_driver_{driver_number}___{period}.json")
    return cars

In [0]:
def get_month_range(year, month):
    today = datetime.today()
    if today.year == year and today.month == month:
        date_start = datetime(year, month, 1)
        date_end = today
    else:
        date_start = datetime(year, month, 1)
        if month == 12:
            next_month = datetime(year + 1, 1, 1)
        else:
            next_month = datetime(year, month + 1, 1)
        date_end = next_month - timedelta(days=1)
    return date_start, date_end

In [0]:
dbutils.widgets.text("year", "")
dbutils.widgets.text("month", "")
dbutils.widgets.text("endpoint", "")

year_param = dbutils.widgets.get("year")
month_param = dbutils.widgets.get("month")
endpoint = dbutils.widgets.get("endpoint")

print(f"Parámetro year recibido: {year_param}")
print(f"Parámetro month recibido: {month_param}")
print(f"Parámetro endpoint recibido: {endpoint}")

current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

year = year_param if year_param else current_year
month = month_param if month_param else current_month
month_str = f"0{month}" if int(month) < 10 else str(month)
date_start, date_end = get_month_range(int(year), int(month))
period = f"{year}_{month_str}"
print({
    "date_start": date_start,
    "date_end": date_end,
    "period": period
})

In [0]:
%sql
SHOW TABLES IN formula_1;

In [0]:
meetings = extract_meetings(date_start, date_end)

In [0]:
for dict in meetings:
    print(dict)
    meeting_key = dict.get("meeting_key", "")
    sessions = extract_sessions(meeting_key, period)
    print(f"Se extrajeron {len(sessions)} sesiones para el meeting_key {meeting_key}")
    for session in sessions:
        session_key = session.get("session_key", "")
        drivers = extract_drivers(meeting_key, session_key, period)
        print(f"Se extrajeron {len(drivers)} conductores para el meeting_key {meeting_key} y session_key {session_key}")
        for driver in drivers:
            driver_number = driver.get("driver_number", "")
            print(f"Se extrajeron datos para el driver_key {driver_number}")
            extract_laps(meeting_key, session_key, driver_number, period)
            extract_cars(session_key, driver_number, period)